# PYTORCH MASTERY — NOTEBOOK 11
## Chapters 59–64: Model Quantization → ONNX and Model Export → TorchScript → Model Serving → REST API Deployment → Docker for PyTorch

Continuing directly from Notebook 10. This is where the course moves from building models into deploying them in production.

> **Reminder**: no PyTorch/internet in this sandbox — code is carefully written but not executed live here.

### PYTORCH MASTERY PROGRESS
Current Chapter: 64 | Current Level: Production Deployment

- [x] Chapters 1–58 *(Notebooks 1–10)*
- [x] Model Quantization
- [x] ONNX and Model Export
- [x] TorchScript
- [x] Model Serving
- [x] REST API Deployment
- [x] Docker for PyTorch


---
# CHAPTER 59 — MODEL QUANTIZATION

## 1. Concept
Quantization reduces a model's numerical precision **beyond** Chapter 29's mixed-precision training (float32/float16) — typically converting weights (and sometimes activations) to **int8** (8-bit integers) — dramatically shrinking model size and speeding up inference on hardware with efficient integer arithmetic support, at the cost of some accuracy (usually small, if done carefully).

## 2. Why Quantization Is Distinct From Mixed Precision
Mixed precision training (Chapter 29) still uses floating-point representations throughout (float16/bfloat16) and is primarily a **training-time** technique. Quantization typically targets **inference-time** deployment specifically, converting to genuinely integer representations (int8, sometimes even int4 for very aggressive LLM quantization) — a fundamentally different numerical representation, not just a smaller float format.

## 3. Post-Training Dynamic Quantization — the Simplest Approach


In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


model = SimpleModel()
model.eval()   # Chapter 23's rule -- ALWAYS eval mode before any inference-focused operation

# Dynamic quantization: weights are quantized to int8 AHEAD of time (once), while
# ACTIVATIONS are quantized on-the-fly DURING inference, dynamically, per batch
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},   # WHICH layer types to quantize -- Linear and LSTM are the most common targets
    dtype=torch.qint8,
)

print("Original model's fc1 weight dtype:", model.fc1.weight.dtype)
print("Quantized model structure:")
print(quantized_model)

def get_model_size_mb(m):
    param_size = sum(p.numel() * p.element_size() for p in m.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in m.buffers())
    return (param_size + buffer_size) / 1e6

print(f"\nOriginal model size: {get_model_size_mb(model):.3f} MB")
print("(The quantized model's ACTUAL size reduction requires the specialized quantized")
print("tensor representation to be inspected directly -- typically a 2-4x reduction for")
print("int8 quantization versus float32, following directly from int8 using 1 byte per")
print("value versus float32's 4 bytes, Chapter 4's element_size() concept.)")


Original model's fc1 weight dtype: torch.float32
Quantized model structure:
SimpleModel(
  (fc1): DynamicQuantizedLinear(in_features=100, out_features=256, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (fc2): DynamicQuantizedLinear(in_features=256, out_features=10, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)

Original model size: 0.114 MB
(The quantized model's ACTUAL size reduction requires the specialized quantized
tensor representation to be inspected directly -- typically a 2-4x reduction for
int8 quantization versus float32, following directly from int8 using 1 byte per
value versus float32's 4 bytes, Chapter 4's element_size() concept.)


/tmp/ipykernel_5135/1727660660.py:20: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


### Line-by-Line Explanation
- `torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` — **dynamic** quantization specifically quantizes weights once (ahead of time, since they're fixed after training) but activations are quantized **dynamically**, computed fresh for each actual input at inference time — this makes dynamic quantization genuinely simple to apply (no calibration data needed) but the per-batch activation quantization step does add a small amount of runtime overhead, a real tradeoff versus static quantization (next).
- `{nn.Linear}` — quantization is applied selectively to specified layer **types** — `nn.Linear` and `nn.LSTM` are the most common, effective targets for dynamic quantization; convolutional layers typically benefit more from **static** quantization instead (below).

## 4. Static Quantization — Calibrated, Typically Faster at Inference


In [ ]:
print("""
STATIC quantization additionally pre-computes fixed quantization parameters for
ACTIVATIONS too (not just weights), using a small CALIBRATION dataset run through
the model once before deployment -- this requires more setup than dynamic
quantization, but avoids activation quantization's PER-BATCH runtime overhead,
typically making static quantization FASTER at actual inference time.

The general workflow:
  1. model.eval()
  2. Insert "observer" modules that RECORD activation statistics during a
     calibration pass (a small, REPRESENTATIVE sample of real data)
  3. Run the calibration data through the model ONCE, letting observers
     collect the statistics needed to determine GOOD quantization parameters
  4. Convert the model to its FINAL, fully-quantized form using those
     calibrated parameters

STATIC quantization works BEST for CNNs (Chapter 35-40); DYNAMIC quantization
is generally preferred for RNN/LSTM/Linear-heavy models (like Transformers'
feed-forward and projection layers) where activation shapes/distributions can
vary more across different inputs, making a FIXED, pre-calibrated activation
quantization scheme less reliable.
""")



STATIC quantization additionally pre-computes fixed quantization parameters for
ACTIVATIONS too (not just weights), using a small CALIBRATION dataset run through
the model once before deployment -- this requires more setup than dynamic
quantization, but avoids activation quantization's PER-BATCH runtime overhead,
typically making static quantization FASTER at actual inference time.

The general workflow:
  1. model.eval()
  2. Insert "observer" modules that RECORD activation statistics during a
     calibration pass (a small, REPRESENTATIVE sample of real data)
  3. Run the calibration data through the model ONCE, letting observers
     collect the statistics needed to determine GOOD quantization parameters
  4. Convert the model to its FINAL, fully-quantized form using those
     calibrated parameters

STATIC quantization works BEST for CNNs (Chapter 35-40); DYNAMIC quantization
is generally preferred for RNN/LSTM/Linear-heavy models (like Transformers'
feed-forward and projection la

## 5. Quantization-Aware Training (QAT) — When Accuracy Loss Is Too Severe


In [ ]:
print("""
Both dynamic and static (POST-training) quantization convert an ALREADY-TRAINED
model afterward -- the model never "experienced" reduced precision during its
own training, so it can sometimes be caught off-guard by the accuracy impact of
quantization, especially for smaller/more sensitive models or very aggressive
quantization schemes (e.g., int4).

QUANTIZATION-AWARE TRAINING (QAT) instead SIMULATES quantization's effects
DURING training itself -- inserting "fake quantization" operations into the
forward pass that mimic int8's reduced precision, while STILL allowing full
float32 gradients to flow during backpropagation (Chapter 13-14). This lets
the model's own training process ADAPT to and compensate for quantization's
precision loss, typically recovering SIGNIFICANTLY more accuracy than
post-training quantization alone, at the cost of requiring a full
(additional) training/fine-tuning pass rather than a one-time, quick
post-hoc conversion.

PRACTICAL GUIDANCE: try post-training dynamic quantization FIRST (cheapest,
fastest to try); if the accuracy drop is unacceptable, try static quantization;
if STILL unacceptable, QAT is the most effective (but most expensive) option.
""")



Both dynamic and static (POST-training) quantization convert an ALREADY-TRAINED
model afterward -- the model never "experienced" reduced precision during its
own training, so it can sometimes be caught off-guard by the accuracy impact of
quantization, especially for smaller/more sensitive models or very aggressive
quantization schemes (e.g., int4).

QUANTIZATION-AWARE TRAINING (QAT) instead SIMULATES quantization's effects
DURING training itself -- inserting "fake quantization" operations into the
forward pass that mimic int8's reduced precision, while STILL allowing full
float32 gradients to flow during backpropagation (Chapter 13-14). This lets
the model's own training process ADAPT to and compensate for quantization's
precision loss, typically recovering SIGNIFICANTLY more accuracy than
post-training quantization alone, at the cost of requiring a full
(additional) training/fine-tuning pass rather than a one-time, quick
post-hoc conversion.

PRACTICAL GUIDANCE: try post-training dyn

### Common Mistakes
- Assuming quantization always causes negligible accuracy loss — the impact genuinely varies by model architecture, task sensitivity, and how aggressively you quantize; always measure the actual accuracy impact on your specific task, never assume it's automatically safe.
- Using static quantization with a calibration dataset that isn't representative of real deployment data — produces poorly-calibrated quantization parameters, hurting both accuracy and any expected speed benefit.
- Applying dynamic quantization to convolutional-heavy models expecting the same benefit static quantization would provide — the layer-type-to-quantization-strategy matching genuinely matters, per Section 4's guidance.
- Forgetting `model.eval()` before any quantization workflow — quantization assumes deterministic, inference-mode behavior (Chapter 23), and skipping this step can produce inconsistent or incorrect calibration/conversion.

### Interview Insight
**Q: You quantize a model to int8 and observe an unacceptable accuracy drop on your validation set. What are your next steps, in order of increasing effort/cost?**
A: First, verify quantization was applied correctly and appropriately for the model's layer types (dynamic quantization for Linear/LSTM-heavy models, static for CNN-heavy models) and that any calibration data used for static quantization was genuinely representative of real deployment inputs. If dynamic quantization was used, try static quantization instead (with a solid calibration dataset), which sometimes recovers more accuracy through more careful, data-informed activation quantization parameters. If accuracy is still unacceptable, Quantization-Aware Training is the most effective (though most expensive) remaining option — it lets the model's own training process directly adapt to and compensate for quantization's precision loss, typically recovering meaningfully more accuracy than any post-training approach, at the cost of requiring additional training compute and time.

### Exercise
1. Apply `torch.quantization.quantize_dynamic` to a small model with `nn.Linear` layers, and compare the model's structure/repr before and after.
2. Explain, in your own words, why dynamic quantization's per-batch activation quantization introduces runtime overhead that static quantization avoids.

### Quiz (Chapter 59)
1. What's the fundamental difference between quantization and Chapter 29's mixed-precision training?
2. What's the key tradeoff between dynamic and static quantization?
3. When would you reach for Quantization-Aware Training specifically?
4. Why does dynamic quantization tend to suit Linear/LSTM-heavy models, while static quantization tends to suit CNNs?


---
# CHAPTER 60 — ONNX AND MODEL EXPORT

## 1. Concept
ONNX (Open Neural Network Exchange) is a **framework-agnostic** format for representing a trained model's computational graph — exporting a PyTorch model to ONNX lets it be run by any ONNX-compatible runtime (ONNX Runtime, TensorRT, various mobile/edge inference engines), independent of PyTorch itself being installed on the deployment target.

## 2. Why This Matters — the Deployment Flexibility Problem
A model trained in PyTorch doesn't necessarily need to be **served** using PyTorch — many production inference environments (mobile devices, embedded systems, certain cloud serving infrastructure, or environments needing multiple frameworks' models served through one unified runtime) benefit from a framework-independent format. ONNX solves exactly this: train once in PyTorch (leveraging PyTorch's excellent development experience), export to ONNX, then deploy using whatever specialized runtime best fits the target environment's specific performance/hardware constraints.

## 3. Exporting a Model to ONNX


In [ ]:
import sys
# Install required ONNX dependencies if not already present
try:
    import onnxscript
except ImportError:
    import subprocess
    print("Installing missing ONNX dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnxscript", "onnx"])

import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


model = SimpleModel()
model.eval()

# ONNX export needs a SAMPLE input -- it TRACES the model's actual computation
# for this specific input shape (see the internal-working note below)
dummy_input = torch.randn(1, 10)

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},   # ALLOW variable batch size
    opset_version=17,   # the ONNX OPERATOR SET version -- affects which ops/features are supported
)

print("Model exported to model.onnx")
print("dynamic_axes is CRITICAL: without it, the exported ONNX model would be HARDCODED")
print("to accept ONLY the exact batch size used during export (here, batch_size=1) --")
print("dynamic_axes marks dimension 0 (the batch dimension) as VARIABLE, letting the")
print("exported model accept ANY batch size at actual inference time, exactly as the")
print("original PyTorch model naturally does.")

/tmp/ipykernel_5135/3963366871.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1005 05:16:00.264000 5135 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model exported to model.onnx
dynamic_axes is CRITICAL: without it, the exported ONNX model would be HARDCODED
to accept ONLY the exact batch size used during export (here, batch_size=1) --
dynamic_axes marks dimension 0 (the batch dimension) as VARIABLE, letting the
exported model accept ANY batch size at actual inference time, exactly as the
original PyTorch model naturally does.


### Line-by-Line Explanation — Tracing vs. Scripting (a Preview of Chapter 61)
- `torch.onnx.export` (in its default mode) works by **tracing**: it runs the model once, on the provided `dummy_input`, and records the exact sequence of tensor operations that specific execution performed — the exported ONNX graph is **this recorded trace**, not a general representation of your Python code's actual logic. This has a genuinely important consequence: any **data-dependent control flow** in your model's `forward()` (an `if` statement whose branch depends on the input's actual values, a variable-length loop) will be "baked in" based on whatever branch happened to execute during tracing, and will **not** correctly generalize to different inputs that would have taken a different branch — Chapter 61's TorchScript scripting mode is the alternative that handles this correctly.
- `opset_version` — ONNX's operator set versions evolve over time (supporting new operations, deprecating old ones); choosing a version compatible with both your model's operations and your target deployment runtime's supported version range is a genuine practical consideration, not an arbitrary number.

## 4. Verifying the Export — Comparing PyTorch and ONNX Runtime Outputs


In [ ]:
import sys
# Install onnxruntime if not already present
try:
    import onnxruntime as ort
except ImportError:
    import subprocess
    print("Installing onnxruntime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnxruntime"])
    import onnxruntime as ort

import numpy as np
import torch

# ONNX Runtime -- a SEPARATE, framework-independent execution engine for running
# ONNX models, completely independent of whether PyTorch is even installed
session = ort.InferenceSession("model.onnx")

test_input = torch.randn(4, 10)   # note: a DIFFERENT batch size (4) than used during export (1) --
                                     # testing that dynamic_axes correctly allows this

# PyTorch's own output, for comparison
with torch.no_grad():
    pytorch_output = model(test_input).numpy()

# ONNX Runtime's output, on the EXACT same input
onnx_output = session.run(None, {"input": test_input.numpy()})[0]

print("PyTorch output shape:", pytorch_output.shape)
print("ONNX Runtime output shape:", onnx_output.shape)
print("Outputs match (within floating-point tolerance):", np.allclose(pytorch_output, onnx_output, atol=1e-5))

PyTorch output shape: (4, 2)
ONNX Runtime output shape: (4, 2)
Outputs match (within floating-point tolerance): True


### Why Verification Matters — Always Confirm the Export, Never Assume
Directly extending the Scikit-learn course's model-persistence verification pattern (Chapter 66 there, `joblib` round-trip checking): **always** verify an ONNX export by comparing outputs against the original PyTorch model on genuinely representative test inputs before trusting the exported model in production — export bugs (unsupported operations silently substituted, tracing artifacts from data-dependent control flow, subtle numerical differences from different underlying kernel implementations) are real and worth catching before deployment, not after.

### Common Mistakes
- Forgetting `dynamic_axes` — produces an ONNX model rigidly locked to the exact batch size (and potentially other dimensions) used during export, breaking at inference time the moment a different batch size is needed.
- Exporting a model with data-dependent control flow using default tracing mode, without verifying the exported graph handles all relevant input variations correctly — a genuinely subtle, easy-to-miss correctness bug (Chapter 61 covers the scripting-mode alternative).
- Not verifying the exported model's output against the original PyTorch model before deploying — skips a cheap, essential correctness check.
- Forgetting `model.eval()` before export — Dropout/BatchNorm's training-mode behavior would be incorrectly baked into the traced graph.

### Interview Insight
**Q: Why does `torch.onnx.export`'s default tracing-based export potentially produce an incorrect model for a network with input-dependent control flow (e.g., a forward pass with an `if` statement based on the input's values)?**
A: Tracing works by running the model once on a specific sample input and recording the literal sequence of tensor operations that particular execution performed — it does not capture the model's Python-level branching *logic* itself, only whichever single execution path the sample input happened to take. If a different input at actual inference time would have taken a different branch of an `if` statement, the traced (and exported) graph has no representation of that alternative branch at all — it will incorrectly continue executing whatever branch was recorded during the original tracing, regardless of what the new input's values would have actually required. This is exactly why verifying export correctness (Section 4) on genuinely representative inputs matters, and why TorchScript's alternative scripting mode (Chapter 61) exists specifically to handle this class of model correctly.

### Exercise
1. Export a small model to ONNX with correct `dynamic_axes`, then load it with `onnxruntime` and verify outputs match PyTorch's on at least two different batch sizes.
2. Explain, in your own words, a concrete example of a model architecture where tracing-based export would silently produce an incorrect result.

### Quiz (Chapter 60)
1. What problem does ONNX solve — why export a model out of PyTorch at all?
2. What does `dynamic_axes` do, and what breaks without it?
3. Why can tracing-based export silently misrepresent a model with data-dependent control flow?
4. What's the essential verification step after any ONNX export, and why should it never be skipped?


---
# CHAPTER 61 — TORCHSCRIPT

## 1. Concept
TorchScript is PyTorch's own **native** serialization and optimization format — unlike ONNX (a cross-framework standard), TorchScript stays within the PyTorch ecosystem, but critically offers a **scripting** mode (in addition to tracing) that correctly captures a model's actual Python control-flow logic, directly solving Chapter 60's tracing limitation.

## 2. Tracing vs. Scripting — the Critical Distinction


In [ ]:
import torch
import torch.nn as nn

class ModelWithControlFlow(nn.Module):
    """Deliberately includes INPUT-DEPENDENT control flow -- exactly the kind
    of model tracing (Chapter 60) handles INCORRECTLY."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 10)
        self.fc2 = nn.Linear(10, 10)

    def forward(self, x):
        if x.sum() > 0:          # THIS branch depends on the INPUT's actual VALUES
            return self.fc1(x)
        else:
            return self.fc2(x)


model = ModelWithControlFlow()
model.eval()

# TRACING: records ONLY the branch taken for THIS specific sample input
positive_input = torch.ones(1, 10)   # sum > 0 -> will trace the fc1 branch
traced_model = torch.jit.trace(model, positive_input)

# SCRIPTING: analyzes the ACTUAL Python source code, correctly preserving BOTH branches
scripted_model = torch.jit.script(model)

# Now test BOTH exported versions on an input that should take the OTHER branch
negative_input = -torch.ones(1, 10)   # sum < 0 -> should correctly use fc2

with torch.no_grad():
    original_output = model(negative_input)
    traced_output = traced_model(negative_input)      # WRONG -- still uses the TRACED (fc1) branch!
    scripted_output = scripted_model(negative_input)    # CORRECT -- genuinely re-evaluates the if statement

print("Original model output (correctly uses fc2 branch):", original_output[:, :3])
print("Traced model output (INCORRECTLY still uses fc1 branch):", traced_output[:, :3])
print("Scripted model output (CORRECTLY uses fc2 branch):", scripted_output[:, :3])
print("\nTraced model matches original:", torch.allclose(original_output, traced_output))
print("Scripted model matches original:", torch.allclose(original_output, scripted_output))


Original model output (correctly uses fc2 branch): tensor([[-0.5856, -0.3852, -0.1732]])
Traced model output (INCORRECTLY still uses fc1 branch): tensor([[-0.1238,  0.0021, -0.2496]])
Scripted model output (CORRECTLY uses fc2 branch): tensor([[-0.5856, -0.3852, -0.1732]])

Traced model matches original: False
Scripted model matches original: True


/tmp/ipykernel_5135/2345287316.py:13: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.sum() > 0:          # THIS branch depends on the INPUT's actual VALUES


### Line-by-Line Explanation — Why This Matters So Much
- `torch.jit.trace(model, positive_input)` — exactly Chapter 60's tracing behavior, now shown producing a **genuinely wrong result** on a different input, concretely demonstrating the exact failure this course has been warning about: the traced model **always** uses the `fc1` branch, regardless of what the actual input's values are, since that's the only branch it ever observed during tracing.
- `torch.jit.script(model)` — instead **parses and analyzes the model's actual Python `forward()` source code** (using a restricted subset of Python that TorchScript can understand and compile), correctly preserving the `if`/`else` logic itself — the scripted model genuinely re-evaluates `x.sum() > 0` on every call, correctly routing to whichever branch the *actual* input requires.
- **The practical rule**: use `torch.jit.script()` (not `torch.jit.trace()`) for any model containing genuinely input-dependent control flow; tracing remains simpler and works correctly for models with no such data-dependent branching (a large fraction of common architectures, including most standard CNNs and Transformers without dynamic depth/routing).

## 3. Saving and Loading TorchScript Models


In [ ]:
scripted_model.save("model_scripted.pt")

loaded_model = torch.jit.load("model_scripted.pt")
with torch.no_grad():
    loaded_output = loaded_model(negative_input)

print("Loaded scripted model output matches:", torch.allclose(loaded_output, scripted_output))
print("CRITICAL ADVANTAGE: a saved TorchScript model can be loaded and run WITHOUT any")
print("Python interpreter at all -- via PyTorch's C++ API (LibTorch) -- genuinely useful")
print("for deployment environments (some embedded/mobile contexts, certain high-performance")
print("serving setups) that specifically need to avoid Python's runtime overhead or simply")
print("don't have a Python environment available at all. This is a capability plain")
print("torch.save(model.state_dict()) (Chapter 70) does NOT provide, since loading a")
print("state_dict still requires the ORIGINAL Python model class definition to be")
print("available to reconstruct the architecture around it.")


Loaded scripted model output matches: True
CRITICAL ADVANTAGE: a saved TorchScript model can be loaded and run WITHOUT any
Python interpreter at all -- via PyTorch's C++ API (LibTorch) -- genuinely useful
for deployment environments (some embedded/mobile contexts, certain high-performance
serving setups) that specifically need to avoid Python's runtime overhead or simply
don't have a Python environment available at all. This is a capability plain
torch.save(model.state_dict()) (Chapter 70) does NOT provide, since loading a
state_dict still requires the ORIGINAL Python model class definition to be
available to reconstruct the architecture around it.


### Common Mistakes
- Using `torch.jit.trace()` on a model with input-dependent control flow without realizing the exported model will silently produce incorrect results on inputs taking a different branch than the one traced — a genuinely dangerous, silent correctness bug, not something that raises an error.
- Assuming TorchScript's scripting mode supports arbitrary Python code — it only supports a **restricted subset** of Python (no arbitrary third-party library calls inside `forward()`, limited dynamic typing) — some models require refactoring to become "scriptable" at all.
- Not testing an exported model (traced OR scripted) on a genuinely diverse set of representative inputs before trusting it — exactly Chapter 60's verification lesson, equally essential here.

### Interview Insight
**Q: You have a PyTorch model with an `if` statement in its `forward()` method whose branch depends on the input tensor's actual values. Why is `torch.jit.trace()` the wrong export choice here, and what should you use instead?**
A: Tracing records only the specific sequence of operations executed for whatever sample input you traced with — it captures no representation of the branching logic itself, only the literal path taken. Any subsequent input that would genuinely need to take the *other* branch will incorrectly still be processed through whichever branch happened to be traced, silently producing wrong results with no error raised. `torch.jit.script()` instead analyzes the model's actual Python source code (within TorchScript's supported subset), correctly preserving conditional logic so the exported model genuinely re-evaluates the condition on every call and routes to the correct branch for each specific input — the correct choice for any model containing data-dependent control flow.

### Exercise
Build a small model with a `for` loop whose iteration count depends on the input's shape (not just its values), and determine (reasoning through it, or testing if you have PyTorch available) whether tracing or scripting handles this correctly.

### Quiz (Chapter 61)
1. What's the fundamental mechanical difference between tracing and scripting?
2. Why did the traced model in this chapter's example produce a wrong (not just suboptimal) result?
3. What capability does a saved TorchScript model have that a plain `state_dict()` save does not?
4. What's a genuine limitation of `torch.jit.script()`'s "restricted subset of Python"?


---
# CHAPTER 62 — MODEL SERVING

## 1. Concept
Model serving is the infrastructure layer that takes a trained (and typically exported/optimized, Chapters 59-61) model and makes it available to answer real-time or batch prediction requests — directly extending the Scikit-learn course's Chapter 69 (API/Deployment Awareness) into PyTorch's specific ecosystem.

## 2. `TorchServe` — PyTorch's Dedicated Serving Framework


In [ ]:
print("""
TorchServe is PyTorch's own purpose-built model serving framework, providing:
  - A REST and gRPC API out of the box (no need to hand-build one, Chapter 63)
  - MODEL VERSIONING and the ability to serve MULTIPLE model versions simultaneously
  - Built-in BATCHING of incoming requests (combining several near-simultaneous
    requests into one forward pass, for GPU efficiency -- Chapter 28's
    batch-size/utilization lesson, applied automatically at the serving layer)
  - Metrics and logging integration

The STANDARD workflow:
  1. Package the model (weights + a "handler" script defining preprocessing/
     postprocessing logic) into a ".mar" (Model Archive) file
  2. Start the TorchServe server, pointing it at the .mar file
  3. Send inference requests to TorchServe's REST/gRPC endpoint

Example handler structure (illustrative):

    class ModelHandler(BaseHandler):
        def preprocess(self, data):
            # convert raw REQUEST data (e.g., JSON) into a model-ready tensor
            ...
        def inference(self, data):
            with torch.inference_mode():   # Chapter 15's rule, STILL applies at serving time
                return self.model(data)
        def postprocess(self, data):
            # convert the model's raw output tensor into a RESPONSE format (e.g., JSON)
            ...
""")



TorchServe is PyTorch's own purpose-built model serving framework, providing:
  - A REST and gRPC API out of the box (no need to hand-build one, Chapter 63)
  - MODEL VERSIONING and the ability to serve MULTIPLE model versions simultaneously
  - Built-in BATCHING of incoming requests (combining several near-simultaneous
    requests into one forward pass, for GPU efficiency -- Chapter 28's
    batch-size/utilization lesson, applied automatically at the serving layer)
  - Metrics and logging integration

The STANDARD workflow:
  1. Package the model (weights + a "handler" script defining preprocessing/
     postprocessing logic) into a ".mar" (Model Archive) file
  2. Start the TorchServe server, pointing it at the .mar file
  3. Send inference requests to TorchServe's REST/gRPC endpoint

Example handler structure (illustrative):

    class ModelHandler(BaseHandler):
        def preprocess(self, data):
            # convert raw REQUEST data (e.g., JSON) into a model-ready tensor
      

## 3. Request Batching at the Serving Layer — Directly Extending Chapter 28


In [ ]:
import time
import threading
import queue

class SimpleBatchingServer:
    """A SIMPLIFIED, illustrative implementation of request batching -- the
    core idea underlying TorchServe's (and most production serving systems')
    automatic batching feature, built from scratch to make the MECHANISM concrete."""

    def __init__(self, model, max_batch_size=8, max_wait_ms=10):
        self.model = model
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.request_queue = queue.Queue()

    def predict_single(self, input_tensor):
        """What an individual CLIENT calls -- appears synchronous to them, but
        internally their request may be BATCHED with other concurrent requests."""
        result_container = {}
        event = threading.Event()
        self.request_queue.put((input_tensor, result_container, event))
        event.wait()   # block until THIS specific request's result is ready
        return result_container["output"]

    def _batch_worker(self):
        """Runs continuously: collects incoming requests for up to max_wait_ms,
        or until max_batch_size requests have arrived, THEN runs ONE batched
        forward pass and distributes results back to each waiting caller."""
        while True:
            batch_items = []
            deadline = time.time() + self.max_wait_ms / 1000
            while len(batch_items) < self.max_batch_size and time.time() < deadline:
                try:
                    remaining = max(0, deadline - time.time())
                    batch_items.append(self.request_queue.get(timeout=remaining))
                except queue.Empty:
                    break

            if not batch_items:
                continue

            inputs = torch.cat([item[0] for item in batch_items], dim=0)   # Chapter 11's cat
            with torch.inference_mode():
                outputs = self.model(inputs)

            for i, (_, result_container, event) in enumerate(batch_items):
                result_container["output"] = outputs[i:i+1]
                event.set()   # WAKE UP this specific client's waiting predict_single() call


print("SimpleBatchingServer illustrates the CORE mechanism: individual, ASYNCHRONOUS")
print("client requests arriving at slightly different times get COLLECTED into a")
print("single batched forward pass -- directly reusing torch.cat (Ch 11) and")
print("torch.inference_mode() (Ch 15) -- then results are DISTRIBUTED back to each")
print("original caller. This is EXACTLY the mechanism TorchServe implements for you")
print("automatically, in a far more robust, production-hardened form.")


SimpleBatchingServer illustrates the CORE mechanism: individual, ASYNCHRONOUS
client requests arriving at slightly different times get COLLECTED into a
single batched forward pass -- directly reusing torch.cat (Ch 11) and
torch.inference_mode() (Ch 15) -- then results are DISTRIBUTED back to each
original caller. This is EXACTLY the mechanism TorchServe implements for you
automatically, in a far more robust, production-hardened form.


### Common Mistakes
- Serving a model that processes exactly one request at a time, with no batching, on hardware where batching would provide substantial GPU utilization benefits (Chapter 28) — leaves real throughput on the table for high-traffic serving scenarios.
- Setting `max_wait_ms` too high — improves batching efficiency (more requests collected per batch) at the direct cost of added per-request latency; this is a genuine, deliberate throughput-vs-latency tradeoff requiring tuning based on the specific application's requirements.
- Forgetting `torch.inference_mode()` inside the serving/inference code path — exactly Chapter 15's lesson, now costing real memory/compute at serving scale across every single request.
- Not using a dedicated serving framework (TorchServe or equivalent) for a genuinely production system, instead hand-rolling fragile serving infrastructure — reasonable for prototyping, risky for production reliability/observability requirements.

### Interview Insight
**Q: Why does request batching at the serving layer improve throughput, and what's the direct tradeoff involved in tuning it?**
A: GPU computation is highly parallel, and processing one request at a time significantly underutilizes that parallelism (directly Chapter 28's low-GPU-utilization lesson) — batching multiple concurrent requests into a single forward pass lets the GPU process them genuinely in parallel rather than serially, dramatically improving overall throughput (requests processed per second) for a given amount of GPU compute. The direct tradeoff is latency: waiting to collect more requests into a larger batch (a longer `max_wait_ms`) improves batching efficiency and throughput, but adds waiting time to every individual request before its result is available — this is a genuine, deliberate tuning decision based on the specific application's latency tolerance versus throughput requirements, not a "more batching is always better" situation.

### Exercise
Trace through `SimpleBatchingServer`'s logic by hand for 3 simulated concurrent requests arriving at slightly different times, and determine whether they'd be batched together or processed separately given a specific `max_wait_ms` value.

### Quiz (Chapter 62)
1. What does TorchServe provide out of the box that you'd otherwise need to build manually?
2. Why does request batching improve serving throughput?
3. What's the direct tradeoff involved in tuning `max_wait_ms` (or an equivalent batching-window parameter)?
4. Why must `torch.inference_mode()` still be used inside a serving handler's inference logic?


---
# CHAPTER 63 — REST API DEPLOYMENT

## 1. Concept
Beyond TorchServe, many teams build a **custom** REST API around a PyTorch model — typically using FastAPI (a modern, popular Python web framework) — directly extending the Scikit-learn course's Chapter 69 conceptual FastAPI sketch, now with a genuine PyTorch model and PyTorch-specific serving considerations.

## 2. A Complete FastAPI Serving Application


In [ ]:
print("""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch

app = FastAPI()

# THE MODEL IS LOADED ONCE, AT STARTUP -- Chapter 68's (Scikit-learn course) lesson,
# now directly applicable: reloading a PyTorch model on every request would be a
# severe, unnecessary latency cost
model = torch.jit.load("model_scripted.pt")   # Chapter 61's TorchScript model
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # Chapter 27
model.to(device)


class PredictionRequest(BaseModel):
    # pydantic automatically VALIDATES incoming request data against this
    # schema -- a more robust, declarative version of the Scikit-learn course's
    # manual validate_input() function (Chapter 68 there).
    features: list[float]


class PredictionResponse(BaseModel):
    prediction: int
    confidence: float


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    if len(request.features) != 10:   # explicit input validation, beyond pydantic's TYPE checking
        raise HTTPException(status_code=400, detail="Expected exactly 10 features")

    input_tensor = torch.tensor([request.features], dtype=torch.float32).to(device)

    with torch.inference_mode():        # Chapter 15's rule -- ALWAYS, at serving time
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=-1)
        confidence, predicted_class = probs.max(dim=-1)

    return PredictionResponse(
        prediction=predicted_class.item(),
        confidence=confidence.item(),
    )


@app.get("/health")
def health_check():
    # A DEDICATED health-check endpoint -- standard practice for any production
    # service, letting load balancers/orchestration systems (Chapter 64's Docker,
    # or Kubernetes) automatically detect and route around an UNHEALTHY instance.
    return {"status": "healthy", "device": str(device)}
""")



from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch

app = FastAPI()

# THE MODEL IS LOADED ONCE, AT STARTUP -- Chapter 68's (Scikit-learn course) lesson,
# now directly applicable: reloading a PyTorch model on every request would be a
# severe, unnecessary latency cost
model = torch.jit.load("model_scripted.pt")   # Chapter 61's TorchScript model
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # Chapter 27
model.to(device)


class PredictionRequest(BaseModel):
    # pydantic automatically VALIDATES incoming request data against this
    # schema -- a more robust, declarative version of the Scikit-learn course's
    # manual validate_input() function (Chapter 68 there).
    features: list[float]


class PredictionResponse(BaseModel):
    prediction: int
    confidence: float


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    if len(request.features) != 10:   # e

### Line-by-Line Explanation
- The model is loaded **once**, at module import time (when the FastAPI app starts), not inside the `predict()` function — exactly the Scikit-learn course's Chapter 68 lesson, now shown in genuine PyTorch serving code.
- `pydantic`'s `BaseModel` classes (`PredictionRequest`/`PredictionResponse`) provide **automatic, declarative** request/response validation — FastAPI uses these to validate incoming JSON against the expected schema and generate clear error messages automatically for malformed requests, a more robust version of the Scikit-learn course's manually-written `validate_input()` function.
- `torch.inference_mode()` wraps the actual model call — this must appear **inside every single request's handling code**, not just once at startup, since inference happens fresh for every incoming request.
- The dedicated `/health` endpoint is genuinely standard production practice — container orchestration systems (Chapter 64) and load balancers use endpoints exactly like this to automatically detect and route traffic away from unhealthy service instances.

## 3. Batch Prediction Endpoint — Handling Multiple Samples in One Request


In [ ]:
print("""
class BatchPredictionRequest(BaseModel):
    samples: list[list[float]]   # a LIST of feature lists -- multiple samples at once


@app.post("/predict_batch")
def predict_batch(request: BatchPredictionRequest):
    input_tensor = torch.tensor(request.samples, dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=-1)
        confidences, predictions = probs.max(dim=-1)

    return {
        "predictions": predictions.tolist(),      # .tolist() -- converting a tensor
        "confidences": confidences.tolist(),        # back to plain Python objects for JSON serialization
    }
""")
print("""
A dedicated batch endpoint (processing MULTIPLE samples in ONE request, via ONE
forward pass) is genuinely more efficient than a client making many separate
single-sample requests -- directly Chapter 62's batching benefit, but achieved
here at the API CONTRACT level (the client explicitly sends a batch) rather
than the serving-INFRASTRUCTURE level (the server implicitly collects
concurrent single requests into a batch) -- both approaches are valid and
often used TOGETHER in a real production system.
""")



class BatchPredictionRequest(BaseModel):
    samples: list[list[float]]   # a LIST of feature lists -- multiple samples at once


@app.post("/predict_batch")
def predict_batch(request: BatchPredictionRequest):
    input_tensor = torch.tensor(request.samples, dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=-1)
        confidences, predictions = probs.max(dim=-1)

    return {
        "predictions": predictions.tolist(),      # .tolist() -- converting a tensor
        "confidences": confidences.tolist(),        # back to plain Python objects for JSON serialization
    }


A dedicated batch endpoint (processing MULTIPLE samples in ONE request, via ONE
forward pass) is genuinely more efficient than a client making many separate
single-sample requests -- directly Chapter 62's batching benefit, but achieved
here at the API CONTRACT level (the client explicitly sends a batch) rather
than the serv

### Common Mistakes
- Reloading the model inside the request-handling function instead of once at startup — a severe, unnecessary latency cost on every single request.
- Not validating input shape/content beyond pydantic's basic type checking — pydantic confirms `features` is a list of floats, but not that it has exactly the right *length* for the model's expected input dimension; explicit additional validation (as shown) remains necessary.
- Returning raw PyTorch tensors directly from an API endpoint instead of converting to plain Python types (`.item()`, `.tolist()`) — tensors aren't directly JSON-serializable, and this produces a serialization error.
- Not including a health-check endpoint — a small omission that becomes a genuine operational problem once the service needs to run behind a load balancer or orchestration system expecting one.

### Interview Insight
**Q: Why is a dedicated `/health` endpoint considered standard, near-mandatory practice for a production model-serving API, even though it doesn't perform any actual prediction?**
A: Production deployments typically run behind load balancers or container orchestration systems (Chapter 64) that need a reliable, lightweight way to determine whether a given service instance is actually healthy and ready to receive traffic — without a dedicated health check, these systems would have no good way to distinguish "temporarily slow but working" from "genuinely crashed or stuck" instances, and might either route traffic to a broken instance (causing failed requests) or fail to route traffic to a genuinely healthy one. A `/health` endpoint gives orchestration systems a fast, low-cost, standardized way to make this determination, enabling automatic detection and routing-around of unhealthy instances, and (in more sophisticated implementations) can also report deeper diagnostic information (like confirming the model actually loaded successfully and which device it's running on, as shown above).

### Exercise
Sketch (in comments, given this sandbox's lack of a running server) a complete FastAPI app for serving a PyTorch image classifier, including request validation, a `/predict` endpoint, a `/health` endpoint, and correct model-loading-once-at-startup structure.

### Quiz (Chapter 63)
1. Why must the model be loaded once at startup, not inside the request handler?
2. What does pydantic's `BaseModel` provide automatically, and what validation does it NOT automatically provide?
3. Why must tensor outputs be converted via `.item()`/`.tolist()` before returning them from an API endpoint?
4. What's the purpose of a dedicated `/health` endpoint?


---
# CHAPTER 64 — DOCKER FOR PYTORCH

## 1. Concept
Docker packages an application (here, a PyTorch serving API) together with its **entire runtime environment** — Python version, exact library versions, system dependencies, even CUDA libraries — into one portable, reproducible container image, directly and completely solving the Scikit-learn course's Chapter 66 environment/version-mismatch concern at the deployment stage.

## 2. A Complete Dockerfile for a PyTorch Serving Application


In [ ]:
dockerfile_content = """
# Start from an OFFICIAL PyTorch base image -- already includes a matched,
# tested combination of PyTorch + CUDA + cuDNN, avoiding the exact version-
# compatibility headaches Chapter 2 covered for LOCAL installation
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime

WORKDIR /app

# Copy and install PYTHON dependencies FIRST, separately from the application
# code -- Docker CACHES each instruction's result; since dependencies change
# far less often than application code, this ordering means Docker can REUSE
# the (often slow) dependency-installation step across many rebuilds, only
# re-running it when requirements.txt itself actually changes
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# THEN copy the actual application code
COPY app/ ./app/
COPY model_scripted.pt .

EXPOSE 8000

# Run the FastAPI app (Chapter 63) via a production-grade ASGI server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print(dockerfile_content)



# Start from an OFFICIAL PyTorch base image -- already includes a matched,
# tested combination of PyTorch + CUDA + cuDNN, avoiding the exact version-
# compatibility headaches Chapter 2 covered for LOCAL installation
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime

WORKDIR /app

# Copy and install PYTHON dependencies FIRST, separately from the application
# code -- Docker CACHES each instruction's result; since dependencies change
# far less often than application code, this ordering means Docker can REUSE
# the (often slow) dependency-installation step across many rebuilds, only
# re-running it when requirements.txt itself actually changes
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# THEN copy the actual application code
COPY app/ ./app/
COPY model_scripted.pt .

EXPOSE 8000

# Run the FastAPI app (Chapter 63) via a production-grade ASGI server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]



In [ ]:
requirements_txt = """
torch==2.3.0
torchvision==0.18.0
fastapi==0.110.0
uvicorn==0.29.0
pydantic==2.6.0
"""
print("requirements.txt (EXACT pinned versions, Ch 66's lesson from the Scikit-learn course):")
print(requirements_txt)

print("""
Building and running this container:

    docker build -t my-pytorch-model:v1 .
    docker run -p 8000:8000 --gpus all my-pytorch-model:v1

--gpus all: passes through GPU access to the container (requires the NVIDIA
Container Toolkit installed on the HOST machine) -- WITHOUT this flag, the
container would only have CPU access, even if the base image includes CUDA
libraries and the host machine genuinely has a GPU.
""")


requirements.txt (EXACT pinned versions, Ch 66's lesson from the Scikit-learn course):

torch==2.3.0
torchvision==0.18.0
fastapi==0.110.0
uvicorn==0.29.0
pydantic==2.6.0


Building and running this container:

    docker build -t my-pytorch-model:v1 .
    docker run -p 8000:8000 --gpus all my-pytorch-model:v1

--gpus all: passes through GPU access to the container (requires the NVIDIA
Container Toolkit installed on the HOST machine) -- WITHOUT this flag, the
container would only have CPU access, even if the base image includes CUDA
libraries and the host machine genuinely has a GPU.



### Line-by-Line Explanation — Why Layer Ordering Matters
- `FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime` — using PyTorch's **official** pre-built base images (rather than starting from a bare Python/Ubuntu image and manually installing PyTorch + CUDA + cuDNN yourself) avoids a substantial class of version-compatibility problems (Chapter 2) entirely — these images are built and tested by the PyTorch team specifically to have compatible versions of every layer.
- `COPY requirements.txt .` / `RUN pip install ...` **before** `COPY app/ ./app/` — this specific ordering is a genuine, important Docker performance practice: Docker builds an image in layers, and caches each layer's result; if application code changes (very frequently, during development) but `requirements.txt` doesn't, Docker can **reuse** the cached (often slow, since installing PyTorch and its dependencies takes real time) dependency-installation layer, only re-running the fast `COPY app/` step — reversing this order would force a full dependency reinstall on every single code change, a substantial, avoidable rebuild-time cost.
- `--gpus all` at `docker run` time — GPU access must be **explicitly** requested when running a container, exactly paralleling Chapter 27's "GPU is never used automatically" lesson, now at the container-runtime level rather than the PyTorch-code level.

## 3. Multi-Stage Builds — Reducing Final Image Size


In [ ]:
multistage_dockerfile = """
# STAGE 1: build/training environment (larger, includes dev tools NOT needed at
# runtime -- compilers, test frameworks, etc.)
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-devel AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt --target=/build/deps

# STAGE 2: FINAL runtime image -- starts FRESH from a smaller base, copies ONLY
# the installed dependencies and application code from Stage 1, discarding
# everything else (build tools, intermediate files, caches) -- the FINAL image
# is dramatically smaller than if everything from Stage 1 were kept
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY app/ ./app/
COPY model_scripted.pt .
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
print(multistage_dockerfile)
print("""
Multi-stage builds are a genuinely common production Docker pattern: separate
the (larger, tool-heavy) BUILD environment from the (smaller, minimal)
RUNTIME environment -- reducing the final deployed image's size, which
directly speeds up container startup/deployment time and reduces the
deployed image's attack surface (fewer unnecessary tools/libraries present
in the final image that could pose a security risk).
""")



# STAGE 1: build/training environment (larger, includes dev tools NOT needed at
# runtime -- compilers, test frameworks, etc.)
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-devel AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt --target=/build/deps

# STAGE 2: FINAL runtime image -- starts FRESH from a smaller base, copies ONLY
# the installed dependencies and application code from Stage 1, discarding
# everything else (build tools, intermediate files, caches) -- the FINAL image
# is dramatically smaller than if everything from Stage 1 were kept
FROM pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY app/ ./app/
COPY model_scripted.pt .
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


Multi-stage builds are a genuinely common production Docker pattern: separate
the (larger, tool-heavy) BUILD environment from the (smaller, minimal

### Common Mistakes
- Copying application code before installing dependencies — causes unnecessary, slow dependency reinstallation on every single code change during development, substantially slowing down the development iteration loop.
- Forgetting `--gpus all` (or the container orchestration system's equivalent configuration) when running on a GPU host — the container silently falls back to CPU-only, even though a GPU is genuinely available on the host, a confusing performance mystery if this isn't the first thing checked.
- Not pinning exact dependency versions in `requirements.txt` — reintroduces exactly the reproducibility risk Chapter 66 (Scikit-learn course) and this course's own Chapter 27/60 discussions specifically warn against, now at the containerization stage where it should be definitively solved.
- Building a single, large image containing both build tools and runtime dependencies when a multi-stage build could produce a meaningfully smaller, more secure final image.

### Interview Insight
**Q: Why does the order of `COPY` and `RUN` instructions in a Dockerfile matter for build performance, and how does Docker's layer caching relate to this?**
A: Docker builds an image as a sequence of cached layers, one per instruction; when rebuilding, Docker reuses a cached layer's result as long as that instruction and everything before it are unchanged, only re-executing from the first genuinely-changed instruction onward. Placing `COPY requirements.txt .` and the subsequent `pip install` **before** `COPY app/ ./app/` means that during iterative development — where application code changes far more frequently than the dependency list — Docker can reuse the cached (and often slow, since installing PyTorch and its dependencies takes real time) dependency-installation layer on every rebuild, only re-running the fast final `COPY` step for the changed code. Reversing this order would invalidate the dependency-installation layer's cache on every single code change, forcing a full, slow dependency reinstall on every rebuild — a substantial, entirely avoidable cost to the development iteration loop.

### Exercise
Write a complete Dockerfile for a hypothetical PyTorch image classification serving app, correctly ordering instructions for cache efficiency and including a multi-stage build to minimize final image size.

### Quiz (Chapter 64)
1. Why does using PyTorch's official base images avoid a category of problems Chapter 2 discussed for local installation?
2. Why must `COPY requirements.txt` + `pip install` come before `COPY app/` in a well-ordered Dockerfile?
3. Why must GPU access be explicitly requested at `docker run` time, and what happens if it's forgotten?
4. What's the benefit of a multi-stage Docker build?

### Chapter 64 Summary
- Docker fully solves the environment reproducibility problem this course has flagged repeatedly (Chapters 2, 27, 60, and the Scikit-learn course's Chapter 66) by packaging the complete runtime environment alongside the application.
- Layer ordering (dependencies before code) and multi-stage builds are genuine, practical performance/size optimizations, not arbitrary conventions.
- GPU access requires explicit configuration at container-runtime, exactly paralleling PyTorch's own "GPU is never automatic" principle from Chapter 27.

---

## END OF NOTEBOOK 11 — What You Should Now Be Able To Do
- Apply dynamic and static post-training quantization appropriately based on model architecture, and know when Quantization-Aware Training is warranted.
- Export a PyTorch model to ONNX correctly (including `dynamic_axes`), verify the export, and explain tracing's data-dependent-control-flow limitation.
- Choose between TorchScript tracing and scripting correctly based on whether a model contains input-dependent control flow, and explain TorchScript's Python-independence advantage.
- Explain the core components of a production model-serving system, including request batching's throughput/latency tradeoff.
- Build a complete FastAPI serving application for a PyTorch model with correct model-loading-once, input validation, and health-check practices.
- Write a well-structured, cache-efficient, multi-stage Dockerfile for a PyTorch serving application, and explain GPU access configuration at the container level.

## Before Moving to Notebook 12 (Chapters 65–71: Model Monitoring in Production, A/B Testing ML Models, Edge Deployment, Mobile Deployment, Distributed Training Fundamentals, Data Parallelism, Gradient Accumulation)
 